# r.manning

This notebook runs the example from r.manning's manual page and visualizes the output.

## Setup

In [ ]:
import os
import subprocess
import sys

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

# Import GRASS packages
import grass.script as gs
import grass.jupyter as gj

Download NLCD dataset:

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

url = "https://www.mrlc.gov/downloads/sciweb1/shared/mrlc/data-bundles/Annual_NLCD_LndCov_2024_CU_C1V1.zip"
nlcd_filename, headers = urllib.request.urlretrieve(url)
with zipfile.ZipFile(nlcd_filename, "r") as zip_ref:
    zip_ref.extractall()
os.remove(nlcd_filename)
nlcd_filename = Path(url).with_suffix(".tif").name

In [ ]:
# create a project
gs.create_project("nlcd", filename=nlcd_filename)
# initialize GRASS session in that project
session = gj.init("nlcd")
gs.run_command("r.external", input=nlcd_filename, output="nlcd")

In [ ]:
session = gj.init("~/grassdata/nc_spm_08_grass7/user1")
gs.run_command("g.region", raster="elevation@PERMANENT")
gs.run_command(
    "r.proj",
    dbase=".",
    project="nlcd",
    mapset="PERMANENT",
    input="nlcd",
    output="nlcd",
    resolution=30,
)

## Create example from the tool's documentation

In [ ]:
r_manning_output = "manning"
gs.run_command("g.region", raster="nlcd")
gs.run_command(
    "r.manning",
    input="nlcd",
    landcover="nlcd",
    output=r_manning_output,
)

In [ ]:
r_manning_map = gj.Map(width=600)
r_manning_map.d_rast(map=r_manning_output)
r_manning_map.d_legend(
    raster=r_manning_output,
    at="2,30,88,92",
    border_color="white",
    flags="tb",
    fontsize=12,
    title="Manning's n",
)
r_manning_map.show()

In [ ]:
from IPython.display import Image

filename = "r_manning.png"
r_manning_map.save(filename)
!mogrify -trim {filename}
!pngquant --ext ".png" -f {filename}
!optipng -o7 {filename}
Image(filename)